# 09 - Long-Horizon `M0/M2/M4` Direct Sweep — 21.03.2026

This notebook is the **feature-focused follow-up** after the `08` direct long-horizon sweep.

It freezes the temporal backbone to the current selected direct architecture and narrows the feature comparison to the modes that still matter:

- `M0`: base temporal core,
- `M2`: base temporal core + system/inertia dynamics,
- `M4`: richer temporal block + static/profile branch.

The point of this notebook is not to re-open architecture tuning. The point is to answer whether the richer inertia/profile feature sets are beneficial at the **longer planning horizons**, even if the gains are modest.

By default it:

- uses the **`A6 = single 64 | L72`** temporal backbone,
- trains with a **fuller budget** than the lighter audit notebooks,
- focuses on the **longer cumulative horizons** where the direct LSTM already showed practical value,
- and exports a clean set of gain tables against `M0`.

In [1]:
# Section 0 - Fast runtime, feature, static-block, and smoke-fit sanity check

from __future__ import annotations

from pathlib import Path
import os
import sys
import platform
import time
import warnings
from typing import Dict, List, Tuple, Optional

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/codex-mplconfig')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
    absl.logging.set_stderrthreshold('error')
except Exception:
    pass

from IPython.display import Markdown, display

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    tf.get_logger().setLevel('ERROR')
except Exception:
    pass
warnings.filterwarnings('ignore', message='.*use_unbounded_threadpool.*')
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
sns.set_theme(style='whitegrid')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

cwd = Path.cwd()
if cwd.name == 'thesis-project':
    PROJECT_ROOT = cwd
elif (cwd / 'thesis-project').exists():
    PROJECT_ROOT = cwd / 'thesis-project'
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / 'data'
FEATURE_DIR = DATA_DIR / 'features'
CLEAN_DIR = DATA_DIR / 'clean'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_METADATA_FILE = FEATURE_DIR / 'feature_metadata.csv'
PORTFOLIO_COVERAGE_FILE = FEATURE_DIR / 'portfolio_coverage.csv'
CAMPUS_PROXY_FILE = CLEAN_DIR / 'campus_building_features_for_models.csv'

RUN_BUILDINGS = ['U05', 'U06', 'LIB', 'U02B', 'SOC', 'U03']
RUN_MODES = ['M0', 'M2', 'M4']
RUN_HORIZONS = [8, 12, 16, 20, 24, 36, 48]
TRAIN_END = pd.Timestamp('2023-12-31 23:00:00')
TEST_START = TRAIN_END + pd.Timedelta(hours=1)
FIT_VALIDATION_FRACTION = 0.10
BATCH_SIZE = 64
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8
LEARNING_RATE = 1e-3
SELECTED_ARCHITECTURE_ID = 'A6'  # Switch to 'A3' if you want the simpler near-tied backbone.

LSTM_BASE_TEMPORAL_FEATURES = [
    'feat_heat_obs',
    'feat_outdoor_temp_c',
    'feat_wind_ms',
    'feat_solar_irradiance_wm2',
    'feat_hour_sin',
    'feat_hour_cos',
    'feat_dow_sin',
    'feat_dow_cos',
]

LSTM_WEATHER_MEMORY_FEATURES = [
    'feat_temp_roll24h',
]

LSTM_SYSTEM_DYNAMIC_FEATURES = [
    'feat_space_heat_active',
    'feat_space_deltaT_c',
    'feat_space_low_deltaT_flag',
    'feat_vent_heat_active',
    'feat_vent_deltaT_c',
    'feat_vent_low_deltaT_flag',
]

LSTM_STATIC_FEATURES_SETB = [
    'stat_heated_area_m2',
    'stat_usage_non_res_share_of_heated',
    'stat_building_age_years',
    'stat_n_points',
    'stat_n_heat_points',
    'stat_n_vent_points',
    'stat_n_dhw_points',
    'stat_vent_class_basic',
    'stat_vent_class_none',
    'stat_vent_class_rich',
    'stat_ventilation_has_heat_recovery',
    'ehr_compactness_ratio',
    'ehr_max_floors',
    'ehr_volume_per_heated_area',
    'stat_missing_heated_area_m2',
    'stat_missing_usage_non_res_share_of_heated',
    'stat_missing_building_age_years',
    'ehr_missing_compactness_ratio',
    'ehr_missing_max_floors',
    'ehr_missing_volume_per_heated_area',
]

MODE_TEMPORAL_FEATURES = {
    'M0': LSTM_BASE_TEMPORAL_FEATURES.copy(),
    'M2': LSTM_BASE_TEMPORAL_FEATURES + LSTM_SYSTEM_DYNAMIC_FEATURES,
    'M4': LSTM_BASE_TEMPORAL_FEATURES + LSTM_WEATHER_MEMORY_FEATURES + LSTM_SYSTEM_DYNAMIC_FEATURES,
}
for mode, cols in MODE_TEMPORAL_FEATURES.items():
    seen = set()
    MODE_TEMPORAL_FEATURES[mode] = [c for c in cols if not (c in seen or seen.add(c))]

ARCHITECTURES = {
    'A3': {
        'architecture_id': 'A3',
        'architecture_label': 'single 64 | L48',
        'lookback_hours': 48,
        'lstm_stack': [64],
        'dropout': 0.0,
        'dense_units': 16,
        'notes': 'Lean single-layer near-tied option from 07.',
    },
    'A6': {
        'architecture_id': 'A6',
        'architecture_label': 'single 64 | L72',
        'lookback_hours': 72,
        'lstm_stack': [64],
        'dropout': 0.0,
        'dense_units': 16,
        'notes': 'Best overall long-horizon planning architecture from 07.',
    },
}
SELECTED_ARCHITECTURE = ARCHITECTURES[SELECTED_ARCHITECTURE_ID]
TOTAL_FITS = len(RUN_BUILDINGS) * len(RUN_MODES) * len(RUN_HORIZONS)

runtime_rows = [
    {'check_name': 'python_executable', 'value': sys.executable, 'status': 'INFO'},
    {'check_name': 'python_architecture', 'value': platform.machine(), 'status': 'PASS' if platform.machine() == 'arm64' else 'WARN'},
    {'check_name': 'tensorflow_version', 'value': tf.__version__, 'status': 'INFO'},
    {'check_name': 'tensorflow_devices', 'value': str(tf.config.list_physical_devices()), 'status': 'INFO'},
    {'check_name': 'gpu_visible_to_tensorflow', 'value': str(tf.config.list_physical_devices('GPU')), 'status': 'PASS' if tf.config.list_physical_devices('GPU') else 'WARN'},
    {'check_name': 'selected_architecture', 'value': SELECTED_ARCHITECTURE_ID, 'status': 'INFO'},
    {'check_name': 'total_planned_fits', 'value': str(TOTAL_FITS), 'status': 'INFO'},
]
runtime_check_df = pd.DataFrame(runtime_rows)
display(runtime_check_df)


def set_all_seeds(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def add_cumulative_targets(df: pd.DataFrame, horizons: List[int]) -> pd.DataFrame:
    out = df.copy()
    heat = out['heat_kwh'].astype(float)
    for horizon in horizons:
        out[f'target_cum_h{horizon}'] = sum(heat.shift(-i) for i in range(horizon))
    return out


def build_sequences(
    df_scaled: pd.DataFrame,
    dynamic_cols: List[str],
    target_col: str,
    split_mask: np.ndarray,
    lookback: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_list, y_list, ts_list = [], [], []
    mask = np.asarray(split_mask, dtype=bool)
    for end_idx in range(lookback, len(df_scaled)):
        if not mask[end_idx]:
            continue
        start_idx = end_idx - lookback
        if not mask[start_idx:end_idx].all():
            continue
        window = df_scaled.iloc[start_idx:end_idx]
        target_val = df_scaled.iloc[end_idx][target_col]
        if window[dynamic_cols].isna().any().any() or pd.isna(target_val):
            continue
        X_list.append(window[dynamic_cols].values.astype('float32'))
        y_list.append(float(target_val))
        ts_list.append(pd.Timestamp(df_scaled.iloc[end_idx]['datetime']))
    if not X_list:
        return (
            np.empty((0, lookback, len(dynamic_cols)), dtype='float32'),
            np.empty((0,), dtype='float32'),
            np.empty((0,), dtype='datetime64[ns]'),
        )
    return (
        np.stack(X_list, axis=0),
        np.array(y_list, dtype='float32'),
        np.array(ts_list, dtype='datetime64[ns]'),
    )


def make_optimizer(learning_rate: float):
    try:
        return tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    except Exception:
        return keras.optimizers.Adam(learning_rate=learning_rate)


def build_lstm_temporal_only_from_spec(n_timesteps: int, n_features: int, spec: Dict, learning_rate: float) -> keras.Model:
    temporal_in = keras.Input(shape=(n_timesteps, n_features), name='temporal_input')
    x = temporal_in
    stack = spec['lstm_stack']
    for layer_idx, units in enumerate(stack, start=1):
        return_sequences = layer_idx < len(stack)
        x = layers.LSTM(units, return_sequences=return_sequences, name=f'lstm_{layer_idx}')(x)
        if return_sequences and spec['dropout'] > 0:
            x = layers.Dropout(spec['dropout'], name=f'dropout_{layer_idx}')(x)
    x = layers.Dense(spec['dense_units'], activation='relu', name='dense_1')(x)
    out = layers.Dense(1, name='output')(x)
    model = keras.Model(inputs=temporal_in, outputs=out, name=f'{spec["architecture_id"]}_temporal_only')
    model.compile(optimizer=make_optimizer(learning_rate), loss='mse', metrics=['mse'])
    return model


def build_lstm_temporal_plus_static_from_spec(
    n_timesteps: int,
    n_dynamic: int,
    n_static_features: int,
    spec: Dict,
    learning_rate: float,
) -> keras.Model:
    temporal_in = keras.Input(shape=(n_timesteps, n_dynamic), name='temporal_input')
    static_in = keras.Input(shape=(n_static_features,), name='static_input')

    x = temporal_in
    stack = spec['lstm_stack']
    for layer_idx, units in enumerate(stack, start=1):
        return_sequences = layer_idx < len(stack)
        x = layers.LSTM(units, return_sequences=return_sequences, name=f'lstm_{layer_idx}')(x)
        if return_sequences and spec['dropout'] > 0:
            x = layers.Dropout(spec['dropout'], name=f'dropout_{layer_idx}')(x)

    s = layers.Dense(32, activation='relu', name='static_dense_1')(static_in)
    s = layers.Dense(16, activation='relu', name='static_dense_2')(s)

    merged = layers.Concatenate(name='concat')([x, s])
    h = layers.Dense(spec['dense_units'], activation='relu', name='merged_dense_1')(merged)
    out = layers.Dense(1, name='output')(h)

    model = keras.Model(inputs=[temporal_in, static_in], outputs=out, name=f'{spec["architecture_id"]}_temporal_plus_static')
    model.compile(optimizer=make_optimizer(learning_rate), loss='mse', metrics=['mse'])
    return model


feature_meta = pd.read_csv(FEATURE_METADATA_FILE)
feature_paths_A = feature_meta.set_index('building')['path_setA'].to_dict()
feature_paths_B = feature_meta.set_index('building')['path_setB'].to_dict()

quick_ok = True
quick_messages = []
try:
    sanity_building = 'U06'
    sanity_horizon = 24
    sanity_target = f'target_cum_h{sanity_horizon}'
    sanity_mode = 'M4'
    sanity_arch = SELECTED_ARCHITECTURE
    sanity_path = PROJECT_ROOT / feature_paths_B[sanity_building]
    if not sanity_path.exists():
        raise FileNotFoundError(sanity_path)

    sanity_df = pd.read_csv(sanity_path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    sanity_df = add_cumulative_targets(sanity_df, [sanity_horizon])

    required_cols = sorted(set(MODE_TEMPORAL_FEATURES[sanity_mode] + LSTM_STATIC_FEATURES_SETB + ['heat_kwh', sanity_target]) - set(sanity_df.columns))
    if required_cols:
        raise ValueError(f'Missing required columns for M4: {required_cols}')

    static_block = sanity_df[LSTM_STATIC_FEATURES_SETB].iloc[[0]]
    if static_block.isna().any().any():
        raise ValueError('Static block contains NaNs in the sanity building.')

    dt = pd.to_datetime(sanity_df['datetime'])
    feature_train_mask = dt <= TRAIN_END
    target_train_mask = dt <= TRAIN_END - pd.Timedelta(hours=sanity_horizon - 1)
    test_mask = dt >= TEST_START

    scaler_X = StandardScaler().fit(sanity_df.loc[feature_train_mask, MODE_TEMPORAL_FEATURES[sanity_mode]].values)
    scaler_y = StandardScaler().fit(sanity_df.loc[target_train_mask, [sanity_target]].dropna().values)
    scaler_static = StandardScaler().fit(static_block.values)

    sanity_scaled = sanity_df.copy()
    sanity_scaled[MODE_TEMPORAL_FEATURES[sanity_mode]] = scaler_X.transform(sanity_scaled[MODE_TEMPORAL_FEATURES[sanity_mode]].values)
    sanity_scaled[f'{sanity_target}_scaled'] = np.nan
    nonnull = sanity_scaled[sanity_target].notna()
    sanity_scaled.loc[nonnull, f'{sanity_target}_scaled'] = scaler_y.transform(sanity_scaled.loc[nonnull, [sanity_target]].values).reshape(-1)

    X_train_full, y_train_full, _ = build_sequences(
        sanity_scaled,
        MODE_TEMPORAL_FEATURES[sanity_mode],
        f'{sanity_target}_scaled',
        target_train_mask.values,
        sanity_arch['lookback_hours'],
    )
    X_test, y_test, _ = build_sequences(
        sanity_scaled,
        MODE_TEMPORAL_FEATURES[sanity_mode],
        f'{sanity_target}_scaled',
        test_mask.values,
        sanity_arch['lookback_hours'],
    )

    if X_train_full.shape[0] < 64 or X_test.shape[0] < 32:
        raise ValueError('Not enough sequences for a meaningful smoke fit.')

    n_val = min(128, max(32, int(round(X_train_full.shape[0] * 0.10))))
    n_val = min(n_val, X_train_full.shape[0] - 1)
    split_at = X_train_full.shape[0] - n_val
    X_train = X_train_full[:split_at][:512]
    y_train = y_train_full[:split_at][:512]
    X_val = X_train_full[split_at:][:128]
    y_val = y_train_full[split_at:][:128]

    static_vec = scaler_static.transform(static_block.values).astype('float32')
    X_train_static = np.repeat(static_vec, repeats=X_train.shape[0], axis=0)
    X_val_static = np.repeat(static_vec, repeats=X_val.shape[0], axis=0)

    set_all_seeds(SEED)
    tf.keras.backend.clear_session()
    smoke_model = build_lstm_temporal_plus_static_from_spec(
        sanity_arch['lookback_hours'],
        len(MODE_TEMPORAL_FEATURES[sanity_mode]),
        static_vec.shape[1],
        sanity_arch,
        LEARNING_RATE,
    )
    smoke_model.fit(
        [X_train, X_train_static],
        y_train,
        validation_data=([X_val, X_val_static], y_val),
        epochs=1,
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    y_pred_scaled = smoke_model.predict([X_val[:64], X_val_static[:64]], verbose=0).reshape(-1, 1)
    y_true = scaler_y.inverse_transform(y_val[:64].reshape(-1, 1)).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
    smoke_rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))

    quick_messages.append(
        f'Smoke fit PASS on {sanity_building}, {sanity_target}, {sanity_mode}, {sanity_arch["architecture_id"]}: '
        f'train_seq={X_train_full.shape[0]}, test_seq={X_test.shape[0]}, 1-epoch val RMSE={smoke_rmse:.2f}'
    )
except Exception as exc:
    quick_ok = False
    quick_messages.append(f'Smoke fit failed: {type(exc).__name__}: {exc}')

status_text = 'PASS' if quick_ok else 'WARN'
display(Markdown(f"""
### Fast sanity check

- **Status:** `{status_text}`
- **Interpretation:** this quick check verifies the runtime, the exact `M0/M2/M4` feature blocks, the static branch availability for `M4`, the cumulative target construction, and one tiny fit with the frozen backbone.
- **Next step if PASS:** start the main run in Section 4a.
- **Next step if WARN:** do **not** start the sweep before fixing the issue below.

{'<br>'.join('- ' + msg for msg in quick_messages)}
"""))

/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,check_name,value,status
0,python_executable,/Users/mihkeluutar/Documents/TalTech/Thesis/th...,INFO
1,python_architecture,arm64,PASS
2,tensorflow_version,2.19.1,INFO
3,tensorflow_devices,"[PhysicalDevice(name='/physical_device:CPU:0',...",INFO
4,gpu_visible_to_tensorflow,"[PhysicalDevice(name='/physical_device:GPU:0',...",PASS
5,selected_architecture,A6,INFO
6,total_planned_fits,126,INFO


I0000 00:00:1774172904.980879 1507056 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1774172904.981049 1507056 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



### Fast sanity check

- **Status:** `PASS`
- **Interpretation:** this quick check verifies the runtime, the exact `M0/M2/M4` feature blocks, the static branch availability for `M4`, the cumulative target construction, and one tiny fit with the frozen backbone.
- **Next step if PASS:** start the main run in Section 4a.
- **Next step if WARN:** do **not** start the sweep before fixing the issue below.

- Smoke fit PASS on U06, target_cum_h24, M4, A6: train_seq=14259, test_seq=7607, 1-epoch val RMSE=650.66


## Section 1 - Experiment design

This run is set up to maximize the chance of seeing whether inertia/profile features help at the longer horizons.

Default setup:

- architecture: **`A6 = single 64 | L72`**
- buildings: `U05, U06, LIB, U02B, SOC, U03`
- modes: **`M0`, `M2`, `M4`**
- cumulative horizons: **`8, 12, 16, 20, 24, 36, 48`**
- seed: `42`
- training budget: **50 epochs**, patience **8**

Interpretation intent:

- `M2 - M0`: isolates the effect of the **system/inertia dynamic channels**,
- `M4 - M2`: asks whether adding the richer temporal-plus-static/profile branch is worth it,
- `M4 - M0`: gives the total gain of the richest direct model in this notebook.

In [2]:
# Section 1a - Metadata, target specs, mode table, and frozen architecture

feature_meta = pd.read_csv(FEATURE_METADATA_FILE)
portfolio_cov = pd.read_csv(PORTFOLIO_COVERAGE_FILE, parse_dates=['first_obs', 'last_obs'])
proxy_df = pd.read_csv(CAMPUS_PROXY_FILE)
feature_paths_A = feature_meta.set_index('building')['path_setA'].to_dict()
feature_paths_B = feature_meta.set_index('building')['path_setB'].to_dict()

missing_A = [b for b in RUN_BUILDINGS if b not in feature_paths_A]
missing_B = [b for b in RUN_BUILDINGS if b not in feature_paths_B]
if missing_A or missing_B:
    raise ValueError(f'Missing feature metadata. setA={missing_A}, setB={missing_B}')

TARGET_SPECS = [
    {
        'target_name': f'target_cum_h{h}',
        'target_family': 'cumulative',
        'horizon_hours': int(h),
    }
    for h in RUN_HORIZONS
]

mode_catalog_df = pd.DataFrame([
    {
        'mode': mode,
        'temporal_features_n': len(MODE_TEMPORAL_FEATURES[mode]),
        'uses_static_branch': bool(mode == 'M4'),
        'temporal_feature_list': ', '.join(MODE_TEMPORAL_FEATURES[mode]),
    }
    for mode in RUN_MODES
])
selected_architecture_df = pd.DataFrame([SELECTED_ARCHITECTURE])
target_specs_df = pd.DataFrame(TARGET_SPECS)

RUNLOG_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_run_log_21032026.csv'
CHECKPOINT_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_checkpoint_21032026.csv'
COVERAGE_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_coverage_21032026.csv'
SUMMARY_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_summary_21032026.csv'
MODE_OVERALL_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_mode_overall_21032026.csv'
GAIN_FILE = RESULTS_DIR / 'm0_m2_m4_long_horizon_gain_vs_M0_21032026.csv'

print('Buildings             :', RUN_BUILDINGS)
print('Modes                 :', RUN_MODES)
print('Horizons              :', RUN_HORIZONS)
print('Selected architecture :', SELECTED_ARCHITECTURE_ID, '-', SELECTED_ARCHITECTURE['architecture_label'])
print('Total fits            :', TOTAL_FITS)

display(target_specs_df)
display(mode_catalog_df)
display(selected_architecture_df[['architecture_id', 'architecture_label', 'lookback_hours', 'lstm_stack', 'dropout', 'dense_units', 'notes']])

Buildings             : ['U05', 'U06', 'LIB', 'U02B', 'SOC', 'U03']
Modes                 : ['M0', 'M2', 'M4']
Horizons              : [8, 12, 16, 20, 24, 36, 48]
Selected architecture : A6 - single 64 | L72
Total fits            : 126


,target_name,target_family,horizon_hours
0,target_cum_h8,cumulative,8
1,target_cum_h12,cumulative,12
2,target_cum_h16,cumulative,16
3,target_cum_h20,cumulative,20
4,target_cum_h24,cumulative,24
5,target_cum_h36,cumulative,36
6,target_cum_h48,cumulative,48


,mode,temporal_features_n,uses_static_branch,temporal_feature_list
0,M0,8,False,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_..."
1,M2,14,False,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_..."
2,M4,15,True,"feat_heat_obs, feat_outdoor_temp_c, feat_wind_..."


,architecture_id,architecture_label,lookback_hours,lstm_stack,dropout,dense_units,notes
0,A6,single 64 | L72,72,[64],0.0,16,Best overall long-horizon planning architectur...


In [3]:
# Section 2 - Load setA / setB frames, build cumulative targets, and define direct splits


def load_feature_frame(path_str: str) -> pd.DataFrame:
    csv_path = PROJECT_ROOT / path_str
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    return pd.read_csv(csv_path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)


features_setA: Dict[str, pd.DataFrame] = {}
features_setB: Dict[str, pd.DataFrame] = {}
split_masks: Dict[str, Dict[str, Dict[str, np.ndarray]]] = {}
coverage_rows = []
all_temporal_A = sorted(set(MODE_TEMPORAL_FEATURES['M0'] + MODE_TEMPORAL_FEATURES['M2']))
all_temporal_B = sorted(set(MODE_TEMPORAL_FEATURES['M4']))

for bldg in RUN_BUILDINGS:
    df_A = load_feature_frame(feature_paths_A[bldg])
    df_B = load_feature_frame(feature_paths_B[bldg])
    df_A = add_cumulative_targets(df_A, RUN_HORIZONS)
    df_B = add_cumulative_targets(df_B, RUN_HORIZONS)

    missing_A = sorted(set(all_temporal_A + ['heat_kwh']) - set(df_A.columns))
    missing_B = sorted(set(all_temporal_B + LSTM_STATIC_FEATURES_SETB + ['heat_kwh']) - set(df_B.columns))
    if missing_A or missing_B:
        raise ValueError(f'{bldg} missing columns. setA={missing_A}, setB={missing_B}')

    dt = pd.to_datetime(df_A['datetime'])
    base_test_mask = dt >= TEST_START
    feature_train_mask = dt <= TRAIN_END

    static_complete = not df_B[LSTM_STATIC_FEATURES_SETB].iloc[[0]].isna().any().any()

    features_setA[bldg] = df_A
    features_setB[bldg] = df_B
    split_masks[bldg] = {}

    row = {
        'building': bldg,
        'rows_total': int(len(df_A)),
        'rows_feature_train': int(feature_train_mask.sum()),
        'rows_test_base': int(base_test_mask.sum()),
        'datetime_start': dt.min(),
        'datetime_end': dt.max(),
        'static_complete_M4': bool(static_complete),
    }
    for spec in TARGET_SPECS:
        target_name = spec['target_name']
        horizon_hours = int(spec['horizon_hours'])
        train_issue_end = TRAIN_END - pd.Timedelta(hours=horizon_hours - 1)
        train_mask = dt <= train_issue_end
        split_masks[bldg][target_name] = {
            'train': train_mask.values,
            'test': base_test_mask.values,
            'feature_train': feature_train_mask.values,
            'train_issue_end': train_issue_end,
        }
        row[f'{target_name}_nonnull_train'] = int(df_A.loc[train_mask, target_name].notna().sum())
        row[f'{target_name}_nonnull_test'] = int(df_A.loc[base_test_mask, target_name].notna().sum())
    coverage_rows.append(row)

coverage_df = pd.DataFrame(coverage_rows).sort_values('building').reset_index(drop=True)
coverage_df.to_csv(COVERAGE_FILE, index=False)
display(coverage_df)
print(f'Saved coverage table to {COVERAGE_FILE}')

,building,rows_total,rows_feature_train,rows_test_base,datetime_start,datetime_end,static_complete_M4,target_cum_h8_nonnull_train,target_cum_h8_nonnull_test,target_cum_h12_nonnull_train,...,target_cum_h16_nonnull_train,target_cum_h16_nonnull_test,target_cum_h20_nonnull_train,target_cum_h20_nonnull_test,target_cum_h24_nonnull_train,target_cum_h24_nonnull_test,target_cum_h36_nonnull_train,target_cum_h36_nonnull_test,target_cum_h48_nonnull_train,target_cum_h48_nonnull_test
0,LIB,22954,14170,8784,2022-05-20 14:00:00,2024-12-31 23:00:00,True,14163,8777,14159,...,14155,8769,14151,8765,14147,8761,14135,8749,14123,8737
1,SOC,22954,14170,8784,2022-05-20 14:00:00,2024-12-31 23:00:00,True,14163,8777,14159,...,14155,8769,14151,8765,14147,8761,14135,8749,14123,8737
2,U02B,22925,14170,8755,2022-05-20 14:00:00,2024-12-30 18:00:00,True,14163,8748,14159,...,14155,8740,14151,8736,14147,8732,14135,8720,14123,8708
3,U03,22925,14170,8755,2022-05-20 14:00:00,2024-12-30 18:00:00,True,14163,8748,14159,...,14155,8740,14151,8736,14147,8732,14135,8720,14123,8708
4,U05,25206,16451,8755,2022-02-14 13:00:00,2024-12-30 18:00:00,True,16444,8748,16440,...,16436,8740,16432,8736,16428,8732,16416,8720,16404,8708
5,U06,25206,16451,8755,2022-02-14 13:00:00,2024-12-30 18:00:00,True,16444,8748,16440,...,16436,8740,16432,8736,16428,8732,16416,8720,16404,8708


Saved coverage table to /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/m0_m2_m4_long_horizon_coverage_21032026.csv


In [4]:
# Section 3 - Scalers and run helpers


temporal_scalers: Dict[Tuple[str, str], StandardScaler] = {}
target_scalers: Dict[Tuple[str, str], StandardScaler] = {}
static_feature_scaler = StandardScaler()

for bldg in RUN_BUILDINGS:
    feature_train_mask = split_masks[bldg][TARGET_SPECS[0]['target_name']]['feature_train']

    for mode in RUN_MODES:
        feature_set = 'setB' if mode == 'M4' else 'setA'
        dynamic_cols = MODE_TEMPORAL_FEATURES[mode]
        scaler = StandardScaler()
        source_df = features_setB[bldg] if feature_set == 'setB' else features_setA[bldg]
        scaler.fit(source_df.loc[feature_train_mask, dynamic_cols].values)
        temporal_scalers[(bldg, mode)] = scaler

    for spec in TARGET_SPECS:
        target_name = spec['target_name']
        target_train_mask = split_masks[bldg][target_name]['train']
        y_train = features_setA[bldg].loc[target_train_mask, [target_name]].dropna().values
        if len(y_train) == 0:
            continue
        scaler_y = StandardScaler()
        scaler_y.fit(y_train)
        target_scalers[(bldg, target_name)] = scaler_y

static_rows = []
for bldg in RUN_BUILDINGS:
    row = features_setB[bldg][LSTM_STATIC_FEATURES_SETB].iloc[[0]].copy()
    if row.isna().any().any():
        continue
    static_rows.append(row.values)
if static_rows:
    static_feature_scaler.fit(np.vstack(static_rows))
else:
    raise ValueError('No complete static rows available for fitting the M4 static scaler.')


def make_internal_fit_split(X: np.ndarray, y: np.ndarray, frac: float):
    n = X.shape[0]
    if n == 0:
        return X, y, X, y
    if frac <= 0 or n < 50:
        return X, y, np.empty((0,) + X.shape[1:], dtype=X.dtype), np.empty((0,), dtype=y.dtype)
    n_val = max(1, int(round(n * frac)))
    n_val = min(n_val, max(1, n - 1))
    split_at = n - n_val
    return X[:split_at], y[:split_at], X[split_at:], y[split_at:]


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    denom = float(np.sum(np.abs(y_true)))
    wape = float(100.0 * np.sum(np.abs(y_true - y_pred)) / denom) if denom > 0 else np.nan
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'wape_pct': wape}


def prepare_scaled_frame(df: pd.DataFrame, building: str, mode: str, dynamic_cols: List[str], target_name: str) -> pd.DataFrame:
    out = df.copy()
    out[dynamic_cols] = temporal_scalers[(building, mode)].transform(out[dynamic_cols].values)
    scaler_y = target_scalers[(building, target_name)]
    nonnull = out[target_name].notna()
    out[f'{target_name}_scaled'] = np.nan
    out.loc[nonnull, f'{target_name}_scaled'] = scaler_y.transform(out.loc[nonnull, [target_name]].values).reshape(-1)
    return out


def build_static_feature_vector(building: str) -> np.ndarray:
    row = features_setB[building][LSTM_STATIC_FEATURES_SETB].iloc[[0]].values
    return static_feature_scaler.transform(row).astype('float32')


def persistence_baseline_prediction(last_heat: np.ndarray, horizon_hours: int) -> np.ndarray:
    return horizon_hours * last_heat


def run_one_case(building: str, target_spec: Dict, mode: str, epochs: int | None = None) -> Dict:
    set_all_seeds(SEED)
    tf.keras.backend.clear_session()

    arch = SELECTED_ARCHITECTURE
    target_name = target_spec['target_name']
    horizon_hours = int(target_spec['horizon_hours'])
    feature_set = 'setB' if mode == 'M4' else 'setA'
    dynamic_cols = MODE_TEMPORAL_FEATURES[mode].copy()
    use_static = mode == 'M4'

    df = features_setB[building].copy() if use_static else features_setA[building].copy()
    masks = split_masks[building][target_name]
    df_scaled = prepare_scaled_frame(df, building, mode, dynamic_cols, target_name)
    target_scaled_col = f'{target_name}_scaled'

    X_train_full, y_train_full, _ = build_sequences(df_scaled, dynamic_cols, target_scaled_col, masks['train'], arch['lookback_hours'])
    X_test, y_test, ts_test = build_sequences(df_scaled, dynamic_cols, target_scaled_col, masks['test'], arch['lookback_hours'])
    X_train, y_train, X_fit_val, y_fit_val = make_internal_fit_split(X_train_full, y_train_full, FIT_VALIDATION_FRACTION)

    row = {
        'building': building,
        'mode': mode,
        'seed': SEED,
        'target_name': target_name,
        'target_family': target_spec['target_family'],
        'horizon_hours': horizon_hours,
        'architecture_id': arch['architecture_id'],
        'architecture_label': arch['architecture_label'],
        'lookback_hours': int(arch['lookback_hours']),
        'lstm_stack': ' -> '.join(str(u) for u in arch['lstm_stack']),
        'dropout': float(arch['dropout']),
        'dense_units': int(arch['dense_units']),
        'feature_set': feature_set,
        'use_static': bool(use_static),
        'n_dynamic_features': len(dynamic_cols),
        'n_static_features': len(LSTM_STATIC_FEATURES_SETB) if use_static else 0,
        'n_train_seq': int(X_train_full.shape[0]),
        'n_fit_val_seq': int(X_fit_val.shape[0]),
        'n_test_seq': int(X_test.shape[0]),
        'train_issue_end': str(masks['train_issue_end']),
        'status': 'ok',
        'notes': arch['notes'],
    }

    if use_static:
        static_vec = build_static_feature_vector(building)
    else:
        static_vec = None

    if X_train.shape[0] == 0 or X_test.shape[0] == 0:
        row.update({'status': 'skipped_insufficient_sequences'})
        for key in ['rmse', 'mae', 'r2', 'wape_pct', 'baseline_wape_pct', 'delta_wape_vs_baseline', 'best_epoch', 'history_epochs_ran']:
            row[key] = np.nan
        return row

    callbacks = []
    validation_data = None
    if X_fit_val.shape[0] > 0:
        callbacks = [
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=0),
        ]

    if use_static:
        X_train_static = np.repeat(static_vec, repeats=X_train.shape[0], axis=0)
        X_val_static = np.repeat(static_vec, repeats=X_fit_val.shape[0], axis=0) if X_fit_val.shape[0] > 0 else np.empty((0, static_vec.shape[1]), dtype='float32')
        X_test_static = np.repeat(static_vec, repeats=X_test.shape[0], axis=0)
        model = build_lstm_temporal_plus_static_from_spec(arch['lookback_hours'], len(dynamic_cols), static_vec.shape[1], arch, LEARNING_RATE)
        if X_fit_val.shape[0] > 0:
            validation_data = ([X_fit_val, X_val_static], y_fit_val)
        history = model.fit(
            [X_train, X_train_static],
            y_train,
            validation_data=validation_data,
            epochs=epochs or EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            callbacks=callbacks,
        )
        y_pred_scaled = model.predict([X_test, X_test_static], verbose=0).reshape(-1, 1)
    else:
        model = build_lstm_temporal_only_from_spec(arch['lookback_hours'], len(dynamic_cols), arch, LEARNING_RATE)
        if X_fit_val.shape[0] > 0:
            validation_data = (X_fit_val, y_fit_val)
        history = model.fit(
            X_train,
            y_train,
            validation_data=validation_data,
            epochs=epochs or EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            callbacks=callbacks,
        )
        y_pred_scaled = model.predict(X_test, verbose=0).reshape(-1, 1)

    scaler_y = target_scalers[(building, target_name)]
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    metrics = compute_metrics(y_true, y_pred)

    feat_obs_idx = dynamic_cols.index('feat_heat_obs')
    temporal_scaler = temporal_scalers[(building, mode)]
    last_heat_scaled = X_test[:, -1, feat_obs_idx]
    feat_mean = temporal_scaler.mean_[feat_obs_idx]
    feat_scale = temporal_scaler.scale_[feat_obs_idx]
    last_heat = last_heat_scaled * feat_scale + feat_mean
    baseline_pred = persistence_baseline_prediction(last_heat.astype(float), horizon_hours)
    baseline_metrics = compute_metrics(y_true, baseline_pred)

    row.update(metrics)
    row['baseline_wape_pct'] = baseline_metrics['wape_pct']
    row['delta_wape_vs_baseline'] = row['wape_pct'] - row['baseline_wape_pct']

    train_losses = history.history.get('loss', [])
    val_losses = history.history.get('val_loss', [])
    row['history_epochs_ran'] = len(train_losses)
    if val_losses:
        row['best_epoch'] = int(np.argmin(val_losses) + 1)
    else:
        row['best_epoch'] = int(np.argmin(train_losses) + 1) if train_losses else np.nan

    return row

## Section 4 - Main run

Default matrix:

- buildings: `6`
- modes: `3`
- cumulative horizons: `7`
- architecture: `1` frozen backbone
- seed: `1`

Total fits: `126`

This is meant to be a few-hour run with a fuller training budget than the lighter audit sweeps.

In [ ]:
# Section 4a - Execute the M0/M2/M4 long-horizon sweep with checkpoint resume

RUN_SWEEP_NOW = True
RESUME_FROM_CHECKPOINT = True


def finalize_run_log(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out['_building_rank'] = out['building'].map({b: i for i, b in enumerate(RUN_BUILDINGS)}).fillna(999).astype(int)
    out['_horizon_rank'] = out['horizon_hours'].fillna(999).astype(int)
    out['_mode_rank'] = out['mode'].map({m: i for i, m in enumerate(RUN_MODES)}).fillna(999).astype(int)
    out = out.sort_values(['_building_rank', '_horizon_rank', '_mode_rank']).drop(columns=['_building_rank', '_horizon_rank', '_mode_rank']).reset_index(drop=True)
    return out


def run_log_matches_current_schema(df: pd.DataFrame) -> bool:
    required_cols = {
        'building', 'mode', 'target_name', 'horizon_hours', 'architecture_id',
        'wape_pct', 'baseline_wape_pct', 'delta_wape_vs_baseline', 'status', 'train_issue_end'
    }
    current_targets = {t['target_name'] for t in TARGET_SPECS}
    current_modes = set(RUN_MODES)
    if df is None or df.empty:
        return False
    if not required_cols.issubset(set(df.columns)):
        return False
    if not set(df['target_name'].dropna().unique()).issubset(current_targets):
        return False
    if not set(df['mode'].dropna().unique()).issubset(current_modes):
        return False
    if 'architecture_id' in df.columns and not set(df['architecture_id'].dropna().unique()).issubset({SELECTED_ARCHITECTURE_ID}):
        return False
    return True


def format_seconds(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours > 0:
        return f'{hours:02d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'


rows = []
completed_keys = set()
if RESUME_FROM_CHECKPOINT and CHECKPOINT_FILE.exists():
    candidate_df = pd.read_csv(CHECKPOINT_FILE)
    if run_log_matches_current_schema(candidate_df):
        rows = candidate_df.to_dict('records')
        ok_df = candidate_df.loc[candidate_df['status'] == 'ok'].copy()
        completed_keys = set(zip(ok_df['building'], ok_df['target_name'], ok_df['mode']))
        print(f'Resuming from checkpoint: {len(completed_keys)} completed fits found.')


total_fits = len(RUN_BUILDINGS) * len(TARGET_SPECS) * len(RUN_MODES)
completed = len(completed_keys)
start_time = time.time()

if RUN_SWEEP_NOW:
    iterator = []
    for bldg in RUN_BUILDINGS:
        for spec in TARGET_SPECS:
            for mode in RUN_MODES:
                iterator.append((bldg, spec, mode))

    progress = tqdm(iterator, desc='m0_m2_m4_long_horizon', total=total_fits) if tqdm is not None else iterator

    for item in progress:
        bldg, spec, mode = item
        fit_key = (bldg, spec['target_name'], mode)
        if fit_key in completed_keys:
            if tqdm is not None:
                progress.set_postfix_str(f'{bldg} {spec["target_name"]} {mode} resumed-skip')
            continue

        row = run_one_case(bldg, spec, mode)
        rows.append(row)
        if row.get('status') == 'ok':
            completed_keys.add(fit_key)
        completed += 1

        elapsed = time.time() - start_time
        rate = elapsed / max(completed, 1)
        eta = rate * max(total_fits - completed, 0)
        status = row.get('status', 'unknown')
        if tqdm is not None:
            progress.set_postfix_str(f'{bldg} {spec["target_name"]} {mode} {status}')
        else:
            print(
                f'[m0_m2_m4] {completed}/{total_fits} fits | building={bldg} target={spec["target_name"]} mode={mode} status={status} | '
                f'elapsed={format_seconds(elapsed)} eta~={format_seconds(eta)}'
            )

        checkpoint_df = finalize_run_log(pd.DataFrame(rows))
        checkpoint_df.to_csv(CHECKPOINT_FILE, index=False)

    run_log_df = finalize_run_log(pd.DataFrame(rows))
    run_log_df.to_csv(RUNLOG_FILE, index=False)
    print(f'Final run log saved to {RUNLOG_FILE}')
else:
    if RUNLOG_FILE.exists():
        candidate_df = pd.read_csv(RUNLOG_FILE)
        if run_log_matches_current_schema(candidate_df):
            run_log_df = finalize_run_log(candidate_df)
            print(f'Loaded existing run log from {RUNLOG_FILE}')
        else:
            run_log_df = pd.DataFrame()
            print(f'[WARN] Existing run log does not match the current 09 schema and was ignored: {RUNLOG_FILE}')
    elif CHECKPOINT_FILE.exists():
        candidate_df = pd.read_csv(CHECKPOINT_FILE)
        if run_log_matches_current_schema(candidate_df):
            run_log_df = finalize_run_log(candidate_df)
            print(f'Loaded checkpoint run log from {CHECKPOINT_FILE}')
        else:
            run_log_df = pd.DataFrame()
            print(f'[WARN] Existing checkpoint does not match the current 09 schema and was ignored: {CHECKPOINT_FILE}')
    else:
        run_log_df = pd.DataFrame()
        print('Run skipped and no compatible existing outputs were found.')

if 'run_log_df' in locals() and not run_log_df.empty:
    display(run_log_df.head())

Resuming from checkpoint: 21 completed fits found.


m0_m2_m4_long_horizon:  55%|█████▍    | 69/126 [43:39<53:31, 56.34s/it, U02B target_cum_h12 M4 ok] 

In [ ]:
# Section 5 - Summaries, gains, and long-horizon comparison plots

from scipy import stats


def mean_ci95(x: pd.Series) -> Tuple[float, float, float]:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    mean = float(arr.mean())
    if len(arr) == 1:
        return mean, mean, mean
    lo, hi = stats.t.interval(0.95, len(arr) - 1, loc=mean, scale=stats.sem(arr))
    return mean, float(lo), float(hi)


if 'run_log_df' not in locals() or run_log_df.empty:
    print('No run log available yet.')
else:
    ok_df = run_log_df.loc[run_log_df['status'] == 'ok'].copy()

    summary_rows = []
    for keys, sub in ok_df.groupby(['target_family', 'target_name', 'horizon_hours', 'mode'], observed=True):
        target_family, target_name, horizon_hours, mode = keys
        w_mean, w_lo, w_hi = mean_ci95(sub['wape_pct'])
        b_mean, b_lo, b_hi = mean_ci95(sub['delta_wape_vs_baseline'])
        summary_rows.append({
            'target_family': target_family,
            'target_name': target_name,
            'horizon_hours': horizon_hours,
            'mode': mode,
            'selected_architecture_id': SELECTED_ARCHITECTURE_ID,
            'n_runs': len(sub),
            'wape_mean': w_mean,
            'wape_ci95_low': w_lo,
            'wape_ci95_high': w_hi,
            'delta_vs_baseline_mean': b_mean,
            'delta_vs_baseline_ci95_low': b_lo,
            'delta_vs_baseline_ci95_high': b_hi,
            'mae_mean': float(sub['mae'].mean()),
            'rmse_mean': float(sub['rmse'].mean()),
            'r2_mean': float(sub['r2'].mean()),
            'baseline_wape_mean': float(sub['baseline_wape_pct'].mean()),
        })
    summary_df = pd.DataFrame(summary_rows).sort_values(['target_family', 'horizon_hours', 'mode']).reset_index(drop=True)

    mode_overall_df = ok_df.groupby(['mode'], observed=True).agg(
        wape_mean=('wape_pct', 'mean'),
        wape_std=('wape_pct', 'std'),
        rmse_mean=('rmse', 'mean'),
        mae_mean=('mae', 'mean'),
        r2_mean=('r2', 'mean'),
        baseline_wape_mean=('baseline_wape_pct', 'mean'),
        delta_vs_baseline_mean=('delta_wape_vs_baseline', 'mean'),
        n_runs=('wape_pct', 'size'),
    ).reset_index().sort_values(['wape_mean', 'rmse_mean']).reset_index(drop=True)
    mode_overall_df['selected_architecture_id'] = SELECTED_ARCHITECTURE_ID
    mode_overall_df['selected_architecture_label'] = SELECTED_ARCHITECTURE['architecture_label']

    gain_rows = []
    for (building, target_name), sub in ok_df.groupby(['building', 'target_name'], observed=True):
        spec = next(s for s in TARGET_SPECS if s['target_name'] == target_name)
        piv = sub.groupby('mode', observed=True)['wape_pct'].mean().to_dict()
        base = float(sub['baseline_wape_pct'].mean())
        if 'M0' not in piv:
            continue
        gain_rows.append({
            'building': building,
            'target_name': target_name,
            'target_family': spec['target_family'],
            'horizon_hours': spec['horizon_hours'],
            'selected_architecture_id': SELECTED_ARCHITECTURE_ID,
            'baseline_wape_pct': base,
            'M0': piv.get('M0', np.nan),
            'M2': piv.get('M2', np.nan),
            'M4': piv.get('M4', np.nan),
            'delta_wape_M2_minus_M0': piv.get('M2', np.nan) - piv.get('M0', np.nan),
            'delta_wape_M4_minus_M0': piv.get('M4', np.nan) - piv.get('M0', np.nan),
            'delta_wape_M4_minus_M2': piv.get('M4', np.nan) - piv.get('M2', np.nan),
            'delta_wape_M0_minus_baseline': piv.get('M0', np.nan) - base,
            'delta_wape_M2_minus_baseline': piv.get('M2', np.nan) - base,
            'delta_wape_M4_minus_baseline': piv.get('M4', np.nan) - base,
        })

    gain_df = pd.DataFrame(gain_rows)
    if not gain_df.empty:
        gain_df = gain_df.merge(
            proxy_df[['building_abv', 'vent_class', 'energy_class', 'ac24', 'night_day_ratio', 'n_vent_points']],
            left_on='building',
            right_on='building_abv',
            how='left',
        ).drop(columns=['building_abv'])

    summary_df.to_csv(SUMMARY_FILE, index=False)
    mode_overall_df.to_csv(MODE_OVERALL_FILE, index=False)
    gain_df.to_csv(GAIN_FILE, index=False)

    display(summary_df)
    display(mode_overall_df)
    if not gain_df.empty:
        display(gain_df.sort_values(['horizon_hours', 'building']).reset_index(drop=True))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    sns.barplot(data=summary_df, x='horizon_hours', y='wape_mean', hue='mode', ax=axes[0], errorbar=None)
    axes[0].set_title(f'Mean WAPE by mode ({SELECTED_ARCHITECTURE_ID})')
    axes[0].set_xlabel('Cumulative horizon [h]')
    axes[0].set_ylabel('Mean WAPE (%)')

    sns.barplot(data=summary_df, x='horizon_hours', y='delta_vs_baseline_mean', hue='mode', ax=axes[1], errorbar=None)
    axes[1].axhline(0.0, color='black', linewidth=1)
    axes[1].set_title(f'Delta WAPE vs persistence ({SELECTED_ARCHITECTURE_ID})')
    axes[1].set_xlabel('Cumulative horizon [h]')
    axes[1].set_ylabel('Delta WAPE (%)')
    plt.tight_layout()
    plt.show()

    if not gain_df.empty:
        focus = gain_df[gain_df['horizon_hours'].isin([12, 24, 36, 48])].copy()
        if not focus.empty:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
            sns.barplot(data=focus, x='building', y='delta_wape_M2_minus_M0', hue='horizon_hours', ax=axes[0], errorbar=None)
            axes[0].axhline(0.0, color='black', linewidth=1)
            axes[0].set_title('Building-level delta WAPE: M2 - M0')
            axes[0].set_ylabel('Delta WAPE (%)')

            sns.barplot(data=focus, x='building', y='delta_wape_M4_minus_M2', hue='horizon_hours', ax=axes[1], errorbar=None)
            axes[1].axhline(0.0, color='black', linewidth=1)
            axes[1].set_title('Building-level delta WAPE: M4 - M2')
            axes[1].set_ylabel('Delta WAPE (%)')
            plt.tight_layout()
            plt.show()

    print('Saved:')
    print('-', RUNLOG_FILE)
    print('-', SUMMARY_FILE)
    print('-', MODE_OVERALL_FILE)
    print('-', GAIN_FILE)

## Transfer-learning note for later

Yes, transfer learning can be tested, but it should be a **separate experiment**, not mixed into this run.

A clean later design would be:

1. group donor buildings by static/profile similarity,
2. pretrain the frozen temporal backbone on donor buildings,
3. fine-tune on the target building,
4. compare against the same target trained from scratch.

For your thesis, the most defensible similarity axes are likely:

- `vent_class`,
- `energy_class`,
- `ac24`,
- `night_day_ratio`,
- `n_vent_points`,
- heated area / building-age style static descriptors.

I would only do that after this direct `M0/M2/M4` sweep, so the pure feature-utility story is clear first.